# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-morad15/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*


I build the feature vector from information available during the February 2026 feature window. The unit of analysis is one client-content pair. The vector uses search-performance signals and basic content characteristics that are available before the March outcome window. Missing numeric values are filled with the median of the corresponding feature, while identifiers and target-related fields are not used as model features.


In [2]:
from google.colab import userdata
from huggingface_hub import login
import duckdb

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found in Colab Secrets."
    )

# Authenticate Hugging Face
login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

# Create DuckDB connection
con = duckdb.connect()

# Configure Hugging Face authentication for DuckDB
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("DuckDB and Hugging Face authentication configured.")

DuckDB and Hugging Face authentication configured.


In [3]:
PERF_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/*.parquet'
)
"""

CONTENT_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
"""

print("Warehouse relations defined.")

Warehouse relations defined.


In [4]:
# Build February-level performance features
feb_features = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(
            COALESCE(sessions_ai, 0)
            + COALESCE(sessions_paid, 0)
            + COALESCE(sessions_direct, 0)
            + COALESCE(sessions_social, 0)
            + COALESCE(sessions_organic, 0)
        ) AS feb_sessions,
        AVG(gsc_sum_position) AS feb_avg_position
    FROM {PERF_REL}
    WHERE report_date BETWEEN '2026-02-01' AND '2026-02-28'
    GROUP BY client_hash_id, content_hash_id
""").df()

# Add basic content features
content_features = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        word_count,
        char_count,
        content_type,
        content_created_date,
        is_published,
        is_deleted
    FROM {CONTENT_REL}
    WHERE is_deleted = FALSE
""").df()

feature_df = feb_features.merge(
    content_features,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Content age using only information available at the end of February
feature_df["content_created_date"] = pd.to_datetime(
    feature_df["content_created_date"]
)

feature_df["content_age_days"] = (
    pd.Timestamp("2026-02-28")
    - feature_df["content_created_date"]
).dt.days

# Model-safe feature columns
feature_columns = [
    "feb_impressions",
    "feb_clicks",
    "feb_sessions",
    "feb_avg_position",
    "word_count",
    "char_count",
    "content_age_days"
]

X = feature_df[feature_columns].copy()

# Median imputation for missing numeric values
for column in feature_columns:
    X[column] = X[column].fillna(X[column].median())

print("Feature vector built successfully.")
print(f"Rows: {len(X):,}")
print(f"Features: {len(feature_columns)}")
print("\nFeatures:")
for feature in feature_columns:
    print(f"- {feature}")

print("\nMissing values after imputation:")
print(X.isna().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector built successfully.
Rows: 321,546
Features: 7

Features:
- feb_impressions
- feb_clicks
- feb_sessions
- feb_avg_position
- word_count
- char_count
- content_age_days

Missing values after imputation:
feb_impressions     0
feb_clicks          0
feb_sessions        0
feb_avg_position    0
word_count          0
char_count          0
content_age_days    0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


| Feature            | Meaning                                                | Missing handling  | Available before prediction? |
| ------------------ | ------------------------------------------------------ | ----------------- | ---------------------------- |
| `feb_impressions`  | Search impressions observed during February            | Median imputation | Yes                          |
| `feb_clicks`       | Search clicks observed during February                 | Median imputation | Yes                          |
| `feb_sessions`     | Aggregated February sessions across available channels | Median imputation | Yes                          |
| `feb_avg_position` | Average observed search position during February       | Median imputation | Yes                          |
| `word_count`       | Number of words in the content                         | Median imputation | Yes                          |
| `char_count`       | Number of characters in the content                    | Median imputation | Yes                          |
| `content_age_days` | Content age measured at the February cutoff            | Median imputation | Yes                          |

All features are constructed from information available no later than February 28, 2026. March outcome information is not used in the feature vector.


In [5]:
# Feature availability and missing-value audit

feature_audit = pd.DataFrame({
    "feature": feature_columns,
    "dtype": X.dtypes.astype(str).values,
    "missing_before_imputation": feature_df[feature_columns].isna().sum().values,
    "missing_after_imputation": X.isna().sum().values
})

display(feature_audit)

print("\nAll model features are numeric:", all(
    pd.api.types.is_numeric_dtype(X[c])
    for c in feature_columns
))

print(
    "All missing values handled:",
    X.isna().sum().sum() == 0
)

,feature,dtype,missing_before_imputation,missing_after_imputation
0,feb_impressions,float64,0,0
1,feb_clicks,float64,0,0
2,feb_sessions,float64,0,0
3,feb_avg_position,float64,0,0
4,word_count,Int64,122903,0
5,char_count,Int64,122903,0
6,content_age_days,float64,17317,0



All model features are numeric: True
All missing values handled: True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


I checked the feature vector for fields that could directly reveal the March outcome or contain information from after the February cutoff.

The target is defined from March 2026 impressions, so March performance fields are excluded from the feature vector. Client and content identifiers are retained only for grouping and traceability and are not used as model features. Fields describing future outcomes or post-cutoff actions are excluded.


In [6]:
# Leakage / privacy audit

forbidden_feature_terms = [
    "mar_",
    "march",
    "outcome",
    "label",
    "declined",
    "change_pct",
    "future"
]

# Columns actually used by the feature vector
feature_names_lower = [c.lower() for c in feature_columns]

leakage_matches = []

for feature in feature_names_lower:
    for term in forbidden_feature_terms:
        if term in feature:
            leakage_matches.append((feature, term))

print("Feature columns checked:")
print(feature_columns)

print("\nPotential leakage matches:")
if leakage_matches:
    for match in leakage_matches:
        print(match)
else:
    print("None found.")

# Check that identifiers are not included in X
identifier_columns = [
    "client_hash_id",
    "content_hash_id"
]

identifier_overlap = [
    c for c in feature_columns
    if c in identifier_columns
]

print("\nIdentifier columns used as model features:")
print(identifier_overlap if identifier_overlap else "None")

# Explicit cutoff check
print("\nFeature cutoff:")
print("2026-02-28")

print("\nLeakage check passed:",
      len(leakage_matches) == 0 and len(identifier_overlap) == 0)

Feature columns checked:
['feb_impressions', 'feb_clicks', 'feb_sessions', 'feb_avg_position', 'word_count', 'char_count', 'content_age_days']

Potential leakage matches:
None found.

Identifier columns used as model features:
None

Feature cutoff:
2026-02-28

Leakage check passed: True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


* **March performance metrics:** excluded because they belong to the outcome window and would leak future information.
* **The outcome label (`declined_20pct`):** excluded because it is the target being predicted, not an input feature.
* **Impression-change percentage:** excluded because it is calculated using February and March performance and therefore contains future outcome information.
* **Client and content hash identifiers:** excluded from model features because they identify entities rather than represent generalizable behavioral or content signals.
* **Post-February content fields:** excluded when they describe information that was not available at the February prediction cutoff.
* **Private URLs and client names:** excluded from the public-facing feature vector to maintain public-safe reporting.


In [7]:
# Final exclusion audit

excluded_fields = {
    "March performance metrics": [
        "mar_impressions",
        "mar_clicks",
        "mar_avg_position"
    ],
    "Target / outcome fields": [
        "declined_20pct",
        "impression_change_pct"
    ],
    "Identifiers": [
        "client_hash_id",
        "content_hash_id"
    ],
    "Private identifying fields": [
        "url",
        "client_name"
    ]
}

for category, fields in excluded_fields.items():
    print(f"\n{category}:")
    for field in fields:
        print(f"  - {field}")

print("\nFinal model feature count:", len(feature_columns))
print("Final feature vector shape:", X.shape)

print("\nW03 leakage/privacy checks completed.")


March performance metrics:
  - mar_impressions
  - mar_clicks
  - mar_avg_position

Target / outcome fields:
  - declined_20pct
  - impression_change_pct

Identifiers:
  - client_hash_id
  - content_hash_id

Private identifying fields:
  - url
  - client_name

Final model feature count: 7
Final feature vector shape: (321546, 7)

W03 leakage/privacy checks completed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.